# 15 - 监督微调 SFT (AI Infra 视角)

本节从 **工程实现** 角度理解 SFT (Supervised Fine-Tuning)：
- SFT 的目标和原理
- nanochat 的 SFT 实现
- 对话数据的 tokenize 和 mask
- 从 HuggingFace 下载模型做 SFT
- 全参数微调 vs LoRA 微调
- 在消费级显卡上的实践

> 参考 nanochat/scripts/chat_sft.py, tokenizer.py, tasks/

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## 1. 核心概念 (30秒版)

```
预训练: 学语言 (大量无标注文本, 预测下一个 token)
   → 产出: 基座模型 (只会续写, 不会对话)

SFT: 学对话 (少量高质量对话数据, 监督学习)
   → 产出: 聊天模型 (理解指令, 回答问题, 知道什么时候停)

关键区别:
  预训练: 所有 token 都参与 loss
  SFT:    只有 assistant 的回复参与 loss (用户的问题被 mask 掉)
```

## 2. 对话数据格式

SFT 的训练数据是一组对话，每条对话包含 user 和 assistant 的多轮消息：

```python
conversation = {
    "messages": [
        {"role": "user", "content": "法国的首都是哪里?"},
        {"role": "assistant", "content": "法国的首都是巴黎。"}
    ]
}
```

Tokenize 后加上 special tokens 变成：

```
<|bos|> <|user_start|> 法国的首都是哪里? <|user_end|> <|assistant_start|> 法国的首都是巴黎。 <|assistant_end|>
```

mask 标注哪些 token 参与 loss：
```
  0        0              0             0          0                  1              1
<|bos|> <|user_start|> 法国...        <|user_end|> <|assistant_start|> 法国...巴黎。  <|assistant_end|>
```

mask=0 的 token 不参与 loss 计算，模型只学习生成 assistant 的回复。

In [ ]:
# 模拟 nanochat tokenizer.render_conversation 的逻辑

def render_conversation_demo(messages):
    """简化版: 展示对话如何被 tokenize 和 mask"""
    ids = []   # token ids (这里用字符串代替)
    mask = []  # 0 = 不参与 loss, 1 = 参与 loss
    
    ids.append("<bos>"); mask.append(0)
    
    for msg in messages:
        if msg["role"] == "user":
            ids.append("<user_start>"); mask.append(0)
            for word in msg["content"].split():
                ids.append(word); mask.append(0)  # 用户内容不参与 loss
            ids.append("<user_end>"); mask.append(0)
        elif msg["role"] == "assistant":
            ids.append("<asst_start>"); mask.append(0)
            for word in msg["content"].split():
                ids.append(word); mask.append(1)  # 助手回复参与 loss!
            ids.append("<asst_end>"); mask.append(1)  # 结束符也参与 loss!
    
    return ids, mask

messages = [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."}
]

ids, mask = render_conversation_demo(messages)

RED = '\033[91m'
GREEN = '\033[92m'
RESET = '\033[0m'

print("Token 和 Mask 可视化:")
print(f"  {RED}红色{RESET} = mask=0 (不学习)  {GREEN}绿色{RESET} = mask=1 (参与 loss)")
print()
for token, m in zip(ids, mask):
    color = GREEN if m == 1 else RED
    print(f"  {color}{token:20s}{RESET} mask={m}")

print(f"\n总 token 数: {len(ids)}, 参与 loss 的: {sum(mask)} ({sum(mask)/len(ids)*100:.0f}%)")

### 为什么 `<|assistant_end|>` 也参与 loss？

因为模型需要学会**主动停止**。如果不监督 `<|assistant_end|>` 的生成，模型在推理时不知道什么时候该停下来，会无限续写。

这就是为什么你之前的 4 层基座模型不会输出 EOS，而 SFT 之后就会了。

## 3. SFT 的 DataLoader

和预训练不同，SFT 的数据是**变长**的（每段对话长度不同）：

```python
# nanochat chat_sft.py 的 collate 逻辑:

# 1. 对话长度不同, 需要 padding
ncols = max(len(ids) for ids, mask in batch) - 1

# 2. inputs: pad 用 <|assistant_end|> (无所谓用什么)
inputs = torch.full((nrows, ncols), pad_token_id)

# 3. targets: pad 用 -1 (ignore_index, 不参与 loss)
targets = torch.full((nrows, ncols), -1)

# 4. 填入实际数据
for i, (ids, mask) in enumerate(batch):
    inputs[i, :n-1] = ids[:-1]
    targets[i, :n-1] = ids[1:]       # 自回归目标: 下一个 token
    targets[i][mask == 0] = -1        # mask 掉 user 部分
```

```
Batch 示例 (3 条对话):

对话 1: [bos user_s What is ... user_e asst_s Paris asst_e] __ __
对话 2: [bos user_s How ...        user_e asst_s The answer is 42 asst_e]
对话 3: [bos user_s Hi user_e asst_s Hello! asst_e] __ __ __ __ __

__ = padding (-1 in targets, 被 loss 忽略)
```

In [ ]:
# SFT loss 计算: 只在 mask=1 的位置算 loss

vocab_size = 1000
B, T = 2, 8

# 模拟 targets: -1 表示不参与 loss
targets = torch.tensor([
    [-1, -1, -1, -1, 50, 100, 200, -1],  # 只有位置 4,5,6 是 assistant 回复
    [-1, -1, 30, 60, 90, -1, -1, -1],     # 只有位置 2,3,4 是 assistant 回复
])

logits = torch.randn(B, T, vocab_size)

# PyTorch 的 cross_entropy 原生支持 ignore_index=-1
loss = F.cross_entropy(
    logits.view(B * T, vocab_size),
    targets.view(B * T),
    ignore_index=-1  # 自动跳过 -1 的位置!
)

# 手动计算验证
valid_mask = targets.view(-1) >= 0
valid_logits = logits.view(B * T, vocab_size)[valid_mask]
valid_targets = targets.view(-1)[valid_mask]
manual_loss = F.cross_entropy(valid_logits, valid_targets)

print(f"总 token 数: {B * T}")
print(f"参与 loss 的 token 数: {valid_mask.sum().item()}")
print(f"ignore_index loss: {loss.item():.4f}")
print(f"手动计算 loss:     {manual_loss.item():.4f}")
print(f"两者一致: {torch.allclose(loss, manual_loss)}")

## 4. nanochat SFT 的训练数据

nanochat 用 `TaskMixture` 混合多个数据集：

```python
train_ds = TaskMixture([
    ARC("ARC-Easy", "train"),       # 2.3K 科学选择题
    ARC("ARC-Challenge", "train"),   # 1.1K 较难科学题
    GSM8K("main", "train"),          # 8K 数学应用题 (带工具调用)
    SmolTalk("train", stop=10000),   # 10K 通用对话
    CustomJSON(identity_file),       # 1K 身份对话 ("你是谁")
    SimpleSpelling(size=300),        # 300 拼写题
    SpellingBee(size=300),           # 300 字母计数题
])  # 总计 ~23K 条对话
```

注意数据量很小！这说明 SFT 不需要海量数据:
- 预训练: 数十亿~数万亿 token
- SFT: 几万条对话就够了
- 关键是数据**质量**而非**数量**

## 5. SFT vs 预训练的关键区别

| | 预训练 | SFT |
|--|--------|-----|
| **数据量** | 数十亿 token | 几万条对话 |
| **数据格式** | 纯文本连续流 | 结构化对话 |
| **Loss 范围** | 所有 token | 只有 assistant 回复 |
| **序列长度** | 固定 (2048) | 变长 (需要 padding) |
| **Epoch** | 通常 < 1 (不重复看数据) | 1~3 epoch |
| **学习率** | 较高 | 较低 (init_lr_frac=0.02) |
| **torch.compile** | 有效 (固定 shape) | 效果差 (变长输入) |
| **目标** | 学习语言和知识 | 学习对话格式和指令遵循 |

In [ ]:
# nanochat SFT 的学习率策略: 线性衰减

import matplotlib.pyplot as plt

num_iterations = 700  # ~23K 条 / 32 per step ≈ 700 步
init_lr_frac = 0.02   # 初始 lr 是预训练 lr 的 2%
base_lr = 0.02        # 预训练的 matrix_lr

initial_lr = base_lr * init_lr_frac  # 0.02 * 0.02 = 0.0004

# 线性衰减到 0
steps = list(range(num_iterations))
lrs = [initial_lr * (1.0 - step / num_iterations) for step in steps]

plt.figure(figsize=(8, 3))
plt.plot(steps, lrs)
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.title('SFT Learning Rate Schedule (Linear Decay)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"预训练 matrix_lr: {base_lr}")
print(f"SFT 初始 lr: {initial_lr} (预训练的 {init_lr_frac*100:.0f}%)")
print(f"\nSFT 用更低的学习率, 避免破坏预训练学到的知识")

## 6. 从 HuggingFace 下载模型做 SFT

不需要自己预训练，可以直接用开源模型做 SFT：

```bash
pip install transformers datasets accelerate peft trl
```

In [ ]:
# 方式一: 全参数 SFT (不用 LoRA)
# 适合小模型 (< 1B) 或大显存 GPU

full_sft_code = '''
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 1. 加载模型和分词器
model_name = "Qwen/Qwen3-0.6B"  # 0.6B 参数, 全参数 SFT 约需 8-9GB 显存
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. 加载数据集 (对话格式)
dataset = load_dataset("HuggingFaceTB/smoltalk", "everyday-conversations", split="train[:5000]")

# 3. 配置训练
training_args = SFTConfig(
    output_dir="./sft_output",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    bf16=True,
    logging_steps=10,
    max_seq_length=512,
)

# 4. 训练!
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()
'''

print("=== 全参数 SFT ===")
print(full_sft_code)
print("特点:")
print("  - 所有参数都可训练")
print("  - 效果最好 (理论上)")
print("  - 显存占用大: ~14 bytes/param")
print("  - 0.6B 模型约 8-9GB, 1.5B 约 21GB")

In [ ]:
# 方式二: LoRA SFT (参数高效微调)
# 适合大模型 + 消费级显卡

lora_sft_code = '''
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# 1. 加载模型 (4-bit 量化)
model_name = "Qwen/Qwen3-0.6B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4-bit 量化
    bnb_4bit_compute_dtype=torch.bfloat16, # 计算用 bf16
    bnb_4bit_quant_type="nf4",            # NormalFloat4 量化
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. 配置 LoRA
lora_config = LoraConfig(
    r=16,                  # LoRA 秩 (越大越强但越慢)
    lora_alpha=32,         # 缩放因子
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # 大约只有 1-2% 可训练

# 3. 加载数据
dataset = load_dataset("HuggingFaceTB/smoltalk", "everyday-conversations", split="train[:5000]")

# 4. 训练
training_args = SFTConfig(
    output_dir="./lora_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,    # 量化后能用更大 batch
    gradient_accumulation_steps=2,
    learning_rate=2e-4,               # LoRA 学习率通常更高
    bf16=True,
    logging_steps=10,
    max_seq_length=512,
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()

# 5. 保存 LoRA adapter (只有几十 MB)
model.save_pretrained("./lora_adapter")
'''

print("=== LoRA SFT (QLoRA) ===")
print(lora_sft_code)
print("特点:")
print("  - 原始模型冻结 + 4-bit 量化")
print("  - 只训练低秩 adapter (~1-2% 参数)")
print("  - 显存极省: 0.6B → ~3GB, 1.5B → ~5GB, 7B → ~10GB")
print("  - 保存的 adapter 只有几十 MB")

## 7. LoRA 原理

```
原始线性层:  y = Wx          (W 是 d×d 的大矩阵, 冻结不动)

LoRA:       y = Wx + BAx    (B 是 d×r, A 是 r×d, r << d)
                  ↑冻结  ↑可训练

例如 d=4096, r=16:
  W: 4096×4096 = 16M 参数 (冻结)
  B: 4096×16   = 65K 参数 (可训练)
  A: 16×4096   = 65K 参数 (可训练)
  总可训练参数: 130K / 16M = 0.8%
```

In [ ]:
# LoRA 的简单实现

class LoRALinear(nn.Module):
    def __init__(self, original_linear, r=16, alpha=32):
        super().__init__()
        self.original = original_linear
        self.original.weight.requires_grad = False  # 冻结原始权重
        
        d_out, d_in = original_linear.weight.shape
        self.lora_A = nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(d_out, r))  # B 初始化为 0
        self.scaling = alpha / r  # 缩放因子
    
    def forward(self, x):
        # 原始输出 + LoRA 增量
        original_out = self.original(x)
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return original_out + lora_out

# 演示
d = 512
r = 16
original = nn.Linear(d, d, bias=False)
lora = LoRALinear(original, r=r, alpha=32)

# 统计参数
total_params = sum(p.numel() for p in lora.parameters())
trainable_params = sum(p.numel() for p in lora.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"原始线性层: {d}×{d} = {d*d:,} 参数")
print(f"LoRA A: {r}×{d} = {r*d:,} 参数")
print(f"LoRA B: {d}×{r} = {d*r:,} 参数")
print(f"\n总参数: {total_params:,}")
print(f"可训练: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
print(f"冻结:   {frozen_params:,} ({frozen_params/total_params*100:.1f}%)")

# 验证初始时 LoRA 不改变输出 (因为 B 初始化为 0)
x = torch.randn(2, d)
diff = (lora(x) - original(x)).abs().max().item()
print(f"\n初始时 LoRA 输出与原始输出的差异: {diff:.10f} (≈0, 因为 B=0)")

## 8. 全参数 vs LoRA: 显存对比

以你的 **RTX 5070 Ti 16GB** 为例：

In [ ]:
# 全参数 vs LoRA 显存对比

def sft_memory_estimate(num_params_b, method="full", lora_pct=0.02):
    """估算 SFT 显存占用 (GB)"""
    P = num_params_b * 1e9
    
    if method == "full":
        model_mem = P * 2 / 1e9        # bf16 参数
        grad_mem = P * 2 / 1e9         # bf16 梯度
        opt_mem = P * 8 / 1e9          # AdamW m+v fp32
        act_mem = 1.5                   # 激活值 (batch_size=1-2)
        total = model_mem + grad_mem + opt_mem + act_mem
        return total, "bf16 全参数"
    
    elif method == "qlora":
        model_mem = P * 0.5 / 1e9      # 4-bit 量化
        trainable_P = P * lora_pct
        grad_mem = trainable_P * 2 / 1e9
        opt_mem = trainable_P * 8 / 1e9
        act_mem = 1.0
        total = model_mem + grad_mem + opt_mem + act_mem
        return total, f"QLoRA (可训练 {lora_pct*100:.0f}%)"

gpu_mem = 16  # RTX 5070 Ti

print(f"GPU: RTX 5070 Ti ({gpu_mem}GB)")
print(f"{'='*65}")
print(f"{'模型':<12} {'方式':<25} {'估算显存':<12} {'能跑?'}")
print(f"{'-'*65}")

for size, name in [(0.6, "0.6B"), (1.5, "1.5B"), (3.0, "3B"), (7.0, "7B")]:
    for method in ["full", "qlora"]:
        mem, desc = sft_memory_estimate(size, method)
        fits = "Yes" if mem < gpu_mem else "No"
        print(f"{name:<12} {desc:<25} {mem:>6.1f} GB     {fits}")
    print()

## 9. LoRA 的关键超参数

```python
LoraConfig(
    r=16,              # 秩: 越大越强但越慢/越费显存
    lora_alpha=32,     # 缩放: 通常设为 2*r
    target_modules=[   # 对哪些层加 LoRA
        "q_proj", "k_proj", "v_proj", "o_proj",   # Attention
        "gate_proj", "up_proj", "down_proj",       # MLP
    ],
    lora_dropout=0.05, # Dropout
)
```

| 参数 | 小 | 大 | 建议 |
|------|-----|-----|------|
| **r** | 8 (省显存) | 64 (更强) | 16 通常够用 |
| **target_modules** | 只加 QV | 加所有线性层 | 加越多效果越好 |
| **learning_rate** | 1e-5 | 5e-4 | LoRA 用 2e-4, 全参数用 2e-5 |

## 10. LoRA 推理: 合并 adapter

训练完成后, LoRA adapter 可以合并回原始模型:

```python
# 训练时: y = Wx + BAx
# 合并后: y = (W + BA)x = W'x
#   → 推理时零额外开销!

# 加载并合并
from peft import PeftModel
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
model = PeftModel.from_pretrained(model, "./lora_adapter")
model = model.merge_and_unload()  # 合并 LoRA 权重到原始模型

# 保存合并后的完整模型
model.save_pretrained("./merged_model")
```

合并后的模型和全参数微调的模型完全等价，推理速度不受影响。

In [ ]:
# 验证 LoRA 合并的数学等价性

d = 256
r = 8

# 原始权重
W = torch.randn(d, d)

# LoRA 权重 (训练后)
A = torch.randn(r, d) * 0.01
B = torch.randn(d, r) * 0.1
alpha = 16
scaling = alpha / r

# 测试输入
x = torch.randn(4, d)

# 方式 1: 分开计算 (训练时)
y1 = x @ W.T + (x @ A.T @ B.T) * scaling

# 方式 2: 合并后计算 (推理时)
W_merged = W + (B @ A) * scaling  # 一次性合并
y2 = x @ W_merged.T

diff = (y1 - y2).abs().max().item()
print(f"分开计算 vs 合并计算的最大差异: {diff:.10f}")
print(f"数学上完全等价!")
print()
print(f"LoRA adapter 大小: {(A.numel() + B.numel()) * 4 / 1024:.1f} KB")
print(f"完整模型大小:      {W.numel() * 4 / 1024:.1f} KB")
print(f"Adapter 只占 {(A.numel() + B.numel()) / W.numel() * 100:.1f}%")

## 11. 在你的 RTX 5070 Ti 上实践

### 方案 A: 用你自己预训练的小模型做全参数 SFT

```bash
# 你的 depth=10 模型训练完后, 仿照 nanochat 的 chat_sft.py
# 准备对话数据 → render_conversation → 训练
# 全参数 SFT, ~200M 模型完全放得下
```

### 方案 B: 下载开源模型做 QLoRA

```bash
# 安装依赖
pip install transformers peft trl datasets bitsandbytes accelerate

# 推荐模型:
#   Qwen/Qwen3-0.6B    → 全参数或 LoRA 都行
#   Qwen/Qwen2.5-1.5B  → QLoRA (~5GB)
#   meta-llama/Llama-3.2-3B → QLoRA (~7GB)
```

### 方案 C: 全参数 SFT 开源 0.6B 模型

```bash
# Qwen3-0.6B 全参数 SFT 约 8-9GB, 你的 16GB 完全够
# 效果比 LoRA 好, 又不会 OOM
# 这是你的甜蜜点!
```

## 12. 面试常见问题

### Q1: SFT 为什么只监督 assistant 的回复？

**答**:
用户的问题是输入条件，不是模型需要学习生成的内容。如果也监督用户部分，模型会浪费容量去"学说用户的话"，偏离"学会回答问题"的目标。

---

### Q2: SFT 会让模型"忘记"预训练的知识吗？

**答**:
会，这叫 **catastrophic forgetting**（灾难性遗忘）。缓解方法：
- 用较低的学习率（nanochat 用预训练 lr 的 2%）
- LoRA 只改一小部分参数，原始知识保持不变
- 训练数据混入通用对话（SmolTalk），防止过度特化

---

### Q3: LoRA 的 r 该设多大？

**答**:
- r=8~16 是最常用的范围
- 简单任务 (格式学习): r=8 足够
- 复杂任务 (知识注入): r=32~64 可能更好
- r 越大效果越接近全参数微调，但显存和训练时间也越多

---

### Q4: 全参数 SFT vs LoRA 效果差多少？

**答**:
- 对于格式学习（学会对话）：LoRA 和全参数几乎一样
- 对于深度知识注入：全参数更好
- 实际差距通常在 1-3% accuracy 以内
- 工业界大量使用 LoRA，因为性价比极高

---

### Q5: 为什么 SFT 不用 torch.compile？

**答**:
SFT 的对话长度不固定（每条对话不一样长），`dynamic=False` 会频繁触发重新编译。nanochat 注释掉了 `torch.compile`：
```python
# model = torch.compile(model, dynamic=True)  # doesn't work super well
```

---

### Q6: QLoRA 的 4-bit 量化会影响训练效果吗？

**答**:
- 推理精度会略有下降（量化损失）
- 但 LoRA 的增量是在 bf16 下训练的，精度有保证
- 实际效果和 bf16 LoRA 差距很小（~0.5%）
- 显存节省巨大（约 4 倍），性价比极高

## 13. 总结速查表

| 主题 | 要点 |
|------|------|
| **SFT 目标** | 教模型对话格式 + 指令遵循 + 何时停止 |
| **数据** | 少量高质量对话 (~10K~100K 条) |
| **Loss Mask** | 只监督 assistant 回复, user 部分 mask 掉 |
| **ignore_index** | targets 中 -1 不参与 loss 计算 |
| **学习率** | 远低于预训练 (通常 1/50) |
| **全参数 SFT** | 效果最好, 显存 ~14 bytes/param |
| **LoRA** | 冻结原模型, 加低秩 adapter, ~1-2% 可训练参数 |
| **QLoRA** | LoRA + 4-bit 量化, 显存极省 |
| **LoRA r** | 8~16 常用, 越大越强但越费资源 |
| **合并** | 训练后 W' = W + BA, 推理无额外开销 |

### 你的 RTX 5070 Ti 16GB 推荐

```
全参数 SFT: ≤ 1B 模型 (Qwen3-0.6B)
QLoRA SFT:  ≤ 7B 模型 (Llama-3.2-3B, Qwen2.5-7B)
```

### nanochat 完整训练流程

```
base_train.py → 预训练 (学语言和知识)
  ↓
mid_train.py  → 中间训练 (过渡)
  ↓
chat_sft.py   → SFT (学对话)  ← 本节
  ↓
chat_rl.py    → RL (优化质量)
```